# Brain Surgery — Bước 0 (E4B): Vocab + PLE Pruning & E2E Multimodal Check

Notebook thiết kế cho **Colab Pro (L4 24GB / A100 40GB)** — chỉ cần bật `HF_TOKEN` trong Colab Secrets và bấm **Run All**:
1. Tải metadata `treadon/gemma4-E4B-it-abliterated` & tự động bù `processor_config.json` / `chat_template.jinja` từ `google/gemma-4-E4B-it` (`E2B` / fallback).
2. Cắt đồng bộ Vocab + PLE `262,144 -> 65,536` (`ID 5: [multimodal]` đăng ký đủ trong `added_tokens`, xóa `tokenizer.model`).
3. Chạy đủ 4 Cổng kiểm chứng (Round-trip, `max_abs_diff`, E2E Generate + Multimodal towers, Export `Q4_K_M` + `--mmproj`).
4. Chạy **E2E `llama-mtmd-cli`** kiểm thử thực tế cả **Vision** (ảnh code) và **Audio** (WAV speech) với `--jinja`.
5. Đẩy checkpoint sạch + `id_map.json` + `pruning_report.json` lên **`haku-nekooo/gemma4-e4b-nuclear-pruned`**.

In [ ]:
!nvidia-smi
!apt-get -qq update && apt-get -qq install -y espeak-ng > /dev/null
!pip install -q --upgrade "transformers>=5.5.0" "safetensors>=0.4.5" "tokenizers>=0.21.0" "accelerate>=1.0.0" "huggingface_hub>=0.28.0" "datasets>=3.0.0" sentencepiece protobuf pillow

In [ ]:
import gc
import json
import os
from pathlib import Path
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/just-haku/brain-surgery.git"
REPO_DIR = Path("/content/brain_surgery")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif not (REPO_DIR / "src" / "surgery").exists():
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass

from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    login()

SOURCE_REPO = "treadon/gemma4-E4B-it-abliterated"  # Fallback: Austriani/Gemma-4-E4B-it-ARA-Abliterated
TARGET_HF_REPO = "haku-nekooo/gemma4-e4b-nuclear-pruned"
TARGET_VOCAB_SIZE = 65536
VI_FLOOR_QUOTA = 4000
CODE_FLOOR_QUOTA = 16000
CORPUS_MAX_DOCS = {"en": 4000, "vi": 5000, "code": 4500}

WORK_DIR = Path("/content/work_step0_e4b")
ORIG_DIR = WORK_DIR / "source_ckpt"
PRUNED_DIR = WORK_DIR / "pruned_ckpt"
CORPUS_CACHE_PATH = WORK_DIR / "corpus_counts.json"
ORIG_LOGITS_PATH = WORK_DIR / "orig_logits_gate2.pt"
GGUF_DIR = WORK_DIR / "gguf_dryrun"
LLAMA_CPP_DIR = Path("/content/llama.cpp")

WORK_DIR.mkdir(parents=True, exist_ok=True)
GGUF_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ready: {SOURCE_REPO} -> {TARGET_HF_REPO} (target_vocab={TARGET_VOCAB_SIZE})")

## 1. Preflight Inspection (Tải metadata & tự bù `processor_config.json` / `chat_template.jinja`)

In [ ]:
from surgery.preflight import download_metadata_only, inspect_checkpoint_metadata

download_metadata_only(SOURCE_REPO, ORIG_DIR, token=HF_TOKEN)
preflight_summary = inspect_checkpoint_metadata(ORIG_DIR, verbose=True)
ORIG_VOCAB_SIZE = preflight_summary["vocab_count"]
assert ORIG_VOCAB_SIZE == 262144 or ORIG_VOCAB_SIZE % 128 == 0
assert not preflight_summary["missing_processor_files"], (
    f"Missing required processor/template files: {preflight_summary['missing_processor_files']}"
)

## 2. Corpus EN + VI + Code & Lọc Từ điển + BPE Merge-Closure

In [ ]:
from transformers import AutoTokenizer
from surgery.config_remap import build_id_map
from surgery.corpus_stats import count_corpus_tokens
from surgery.vocab_filter import collect_mandatory_whitelist, select_hybrid_vocab

orig_tok = AutoTokenizer.from_pretrained(str(ORIG_DIR), use_fast=True)
corpus_counts = count_corpus_tokens(
    orig_tok,
    max_docs_per_lang=CORPUS_MAX_DOCS,
    cache_path=CORPUS_CACHE_PATH,
)
del orig_tok
gc.collect()

src_config = json.loads((ORIG_DIR / "config.json").read_text(encoding="utf-8"))
src_tok_json = json.loads((ORIG_DIR / "tokenizer.json").read_text(encoding="utf-8"))
src_tok_cfg = (
    json.loads((ORIG_DIR / "tokenizer_config.json").read_text(encoding="utf-8"))
    if (ORIG_DIR / "tokenizer_config.json").exists()
    else None
)
src_gen_cfg = (
    json.loads((ORIG_DIR / "generation_config.json").read_text(encoding="utf-8"))
    if (ORIG_DIR / "generation_config.json").exists()
    else None
)
chat_tpl = (
    (ORIG_DIR / "chat_template.jinja").read_text(encoding="utf-8")
    if (ORIG_DIR / "chat_template.jinja").exists()
    else None
)

vocab_by_id = {int(tid): tok for tok, tid in src_tok_json["model"]["vocab"].items()}
for entry in src_tok_json.get("added_tokens") or []:
    vocab_by_id.setdefault(int(entry["id"]), entry["content"])

whitelist_ids = collect_mandatory_whitelist(
    vocab_by_id=vocab_by_id,
    config=src_config,
    tokenizer_config=src_tok_cfg,
    generation_config=src_gen_cfg,
    tokenizer_json=src_tok_json,
    chat_template=chat_tpl,
)
kept_old_ids, pruned_merges_raw, pruning_report = select_hybrid_vocab(
    vocab_by_id=vocab_by_id,
    raw_merges=src_tok_json["model"]["merges"],
    whitelist_ids=whitelist_ids,
    en_counts=corpus_counts["en"],
    vi_counts=corpus_counts["vi"],
    code_counts=corpus_counts["code"],
    target_size=TARGET_VOCAB_SIZE,
    vi_floor=VI_FLOOR_QUOTA,
    code_floor=CODE_FLOOR_QUOTA,
)
id_map = build_id_map(kept_old_ids)
(WORK_DIR / "id_map.json").write_text(
    json.dumps({"kept_old_ids": kept_old_ids, "old_to_new": {str(k): v for k, v in id_map.items()}}),
    encoding="utf-8",
)
print(json.dumps(pruning_report, indent=2, ensure_ascii=False))

## 3. Tải Weights E4B, Capture Logits Gốc (2A) & Low-RAM Shard Slicing

In [ ]:
from huggingface_hub import snapshot_download
from surgery.config_remap import remap_and_save_metadata
from surgery.shard_slicer import slice_checkpoint_shards
from surgery.verify import capture_orig_logits

snapshot_download(repo_id=SOURCE_REPO, local_dir=str(ORIG_DIR), token=HF_TOKEN)
gate2_capture_info = capture_orig_logits(
    orig_dir=ORIG_DIR,
    kept_old_ids=kept_old_ids,
    save_path=ORIG_LOGITS_PATH,
)
print("Phase 2A:", gate2_capture_info)

slicing_report = slice_checkpoint_shards(
    src_dir=ORIG_DIR,
    dst_dir=PRUNED_DIR,
    kept_old_ids=kept_old_ids,
    orig_vocab_size=ORIG_VOCAB_SIZE,
)
remap_report = remap_and_save_metadata(
    src_dir=ORIG_DIR,
    dst_dir=PRUNED_DIR,
    kept_old_ids=kept_old_ids,
    pruned_merges_raw=pruned_merges_raw,
    id_map=id_map,
)
assert not remap_report["missing_processor_files"], f"Missing in PRUNED_DIR: {remap_report['missing_processor_files']}"

full_report = {
    "source_repo": SOURCE_REPO,
    "preflight": preflight_summary,
    "pruning": pruning_report,
    "slicing": slicing_report,
    "remap": remap_report,
}
(PRUNED_DIR / "id_map.json").write_text((WORK_DIR / "id_map.json").read_text(encoding="utf-8"), encoding="utf-8")
(PRUNED_DIR / "pruning_report.json").write_text(json.dumps(full_report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Sliced {slicing_report['saved_gb']} GB | Peak RSS={slicing_report['peak_rss_gb']} GB | PLE={slicing_report['has_ple_sliced']} | Injected={remap_report['injected_added_tokens']}")

## 4. Cổng Kiểm chứng 1, 2 & 3 (Tokenizer, Logits & `Gemma4ForConditionalGeneration`)

In [ ]:
from surgery.verify import run_gate1_tokenizer, run_gate2_and_gate3

gate1_res = run_gate1_tokenizer(PRUNED_DIR, id_map=id_map)
gate23_res = run_gate2_and_gate3(
    pruned_dir=PRUNED_DIR,
    id_map=id_map,
    orig_logits_path=ORIG_LOGITS_PATH,
)
print(json.dumps({"gate1": gate1_res, "gate23": gate23_res}, indent=2, ensure_ascii=False))

## 5. Cổng 4 — Export GGUF (`Q4_K_M` + `--mmproj`) & Build `llama-mtmd-cli`

In [ ]:
if (LLAMA_CPP_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(LLAMA_CPP_DIR), "pull", "--ff-only"], check=True)
elif not LLAMA_CPP_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(LLAMA_CPP_DIR)], check=True)

!pip install -q --no-deps gguf
!cmake -B {LLAMA_CPP_DIR}/build -S {LLAMA_CPP_DIR} -DGGML_CUDA=OFF
!cmake --build {LLAMA_CPP_DIR}/build --config Release --target llama-quantize llama-cli llama-mtmd-cli -j$(nproc)

In [ ]:
CONVERT_PY = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"
QUANTIZE_BIN = LLAMA_CPP_DIR / "build" / "bin" / "llama-quantize"
CLI_BIN = LLAMA_CPP_DIR / "build" / "bin" / "llama-cli"
MTMD_BIN = LLAMA_CPP_DIR / "build" / "bin" / "llama-mtmd-cli"

main_bf16 = GGUF_DIR / "gemma4-e4b-step0-bf16.gguf"
main_q4 = GGUF_DIR / "gemma4-e4b-step0-Q4_K_M.gguf"
mmproj_bf16 = GGUF_DIR / "mmproj-e4b-step0-bf16.gguf"

subprocess.run(["python", str(CONVERT_PY), str(PRUNED_DIR), "--outfile", str(main_bf16), "--outtype", "bf16"], check=True)
subprocess.run([str(QUANTIZE_BIN), str(main_bf16), str(main_q4), "Q4_K_M"], check=True)
subprocess.run(["python", str(CONVERT_PY), str(PRUNED_DIR), "--mmproj", "--outfile", str(mmproj_bf16), "--outtype", "bf16"], check=True)

cli_help = subprocess.run([str(CLI_BIN), "--help"], capture_output=True, text=True).stdout
cli_st = ["--single-turn"] if "--single-turn" in cli_help else (["-no-cnv"] if "-no-cnv" in cli_help else [])
cli_proc = subprocess.run(
    [str(CLI_BIN), "-m", str(main_q4), "-p", "Viết hàm Python kiểm tra số nguyên tố:", "-n", "48", "--no-display-prompt", *cli_st],
    stdin=subprocess.DEVNULL,
    capture_output=True,
    text=True,
    check=True,
)
q4_gb = main_q4.stat().st_size / (1024**3)
mmproj_gb = mmproj_bf16.stat().st_size / (1024**3)
print(f"Core Brain Q4_K_M: {q4_gb:.3f} GB | mmproj: {mmproj_gb:.3f} GB | Total: {q4_gb + mmproj_gb:.3f} GB")
print("llama-cli preview:\n", cli_proc.stdout[:400])

## 6. E2E Multimodal Verification (`llama-mtmd-cli` Vision + Audio với `--jinja`)

In [ ]:
from PIL import Image, ImageDraw, ImageFont

img_path = GGUF_DIR / "test_code.png"
img = Image.new("RGB", (640, 180), "white")
draw = ImageDraw.Draw(img)
try:
    font = ImageFont.load_default(size=28)
except TypeError:
    font = ImageFont.load_default()
draw.text((24, 48), "def square(x: int) -> int:\n    return x * x", fill="black", font=font)
img.save(img_path)

wav_path = GGUF_DIR / "test_speech.wav"
subprocess.run(
    ["espeak-ng", "-s", "140", "-w", str(wav_path), "Hello Gemma four, what is seven plus five?"],
    check=True,
)

help_txt = subprocess.run([str(MTMD_BIN), "--help"], capture_output=True, text=True).stdout
img_flag = "--image" if "--image" in help_txt else "--media"
aud_flag = "--audio" if "--audio" in help_txt else "--media"
extra_flags = ["--no-warmup"] if "--no-warmup" in help_txt else []
if "--single-turn" in help_txt:
    extra_flags.append("--single-turn")

def _run_mtmd(flag: str, media_file: Path, prompt: str) -> str:
    cmd = [
        str(MTMD_BIN),
        "-m", str(main_q4),
        "--mmproj", str(mmproj_bf16),
        flag, str(media_file),
        "-p", prompt,
        "-n", "300",
        "--jinja",
        *extra_flags,
    ]
    proc = subprocess.run(cmd, stdin=subprocess.DEVNULL, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"llama-mtmd-cli failed ({flag}):\n{proc.stderr[-1500:]}")
    out = proc.stdout.strip()
    assert out, f"Empty output from llama-mtmd-cli ({flag})"
    return out

vision_out = _run_mtmd(img_flag, img_path, "Read the Python function in this image and state what square(7) returns.")
print("=== E2E VISION (--jinja) ===\n", vision_out[:600])

audio_out = _run_mtmd(aud_flag, wav_path, "Transcribe the audio and answer the math question.")
print("\n=== E2E AUDIO (--jinja) ===\n", audio_out[:600])

full_report["gates"] = {
    "gate1": gate1_res,
    "gate23": gate23_res,
    "gate4": {"main_q4_km_gb": round(q4_gb, 3), "mmproj_exported": True, "mmproj_gb": round(mmproj_gb, 3)},
    "e2e_multimodal": {"vision_preview": vision_out[:300], "audio_preview": audio_out[:300]},
}
(PRUNED_DIR / "pruning_report.json").write_text(json.dumps(full_report, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved updated pruning_report.json with E2E Vision & Audio results.")

## 7. Đẩy E4B Pruned Checkpoint lên Hugging Face Private Repo

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=TARGET_HF_REPO, repo_type="model", private=True, exist_ok=True)
api.upload_folder(
    folder_path=str(PRUNED_DIR),
    repo_id=TARGET_HF_REPO,
    repo_type="model",
    commit_message=f"Step 0 E4B: pruned vocab to {TARGET_VOCAB_SIZE} + passed all 4 gates + E2E mtmd Vision/Audio",
)
print(f"Uploaded E4B Step 0 checkpoint -> https://huggingface.co/{TARGET_HF_REPO}")